# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [6]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [7]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [8]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-cou

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [9]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [10]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [11]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
htt

In [12]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [13]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/'}]}

In [14]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [15]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter/X profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [16]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'brand assets', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn resources', 'url': 'https://huggingface.co/learn'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'API Endpoints', 'url': 'https://endpoints.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [17]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
MiniMaxAI/MiniMax-M2.1
Updated
4 days ago
•
141k
•
702
zai-org/GLM-4.7
Updated
8 days ago
•
30.5k
•
1.29k
tencent/HY-MT1.5-1.8B
Updated
1 day ago
•
847
•
365
Qwen/Qwen-Image-Edit-2511
Updated
8 days ago
•
32.1k
•
583
LiquidAI/LFM2-2.6B-Exp
Updated
5 days ago
•
4.91k
•
272
Browse 2M+ models
Spaces
Running
Featured
3.29k
Wan2.2 Animate
👁
3.29k
Wan2.2 Animate
Running
on
Zero
Featured
662
TRELLIS.2
🏢
662
High-fidelity 3D Generation from images
Running
on
CPU Upgrade
330
Omni Image Editor
🖼
330
Image edit, text to image, face swap, image 

In [19]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

#brochure_system_prompt = """
#You are an assistant that analyzes the contents of several relevant pages from a company website
#and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
#Respond in markdown without code blocks.
#Include details of company culture, customers and careers/jobs if you have the information.
#"""


In [20]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [21]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nMiniMaxAI/MiniMax-M2.1\nUpdated\n4 days ago\n•\n141k\n•\n702\nzai-org/GLM-4.7\nUpdated\n8 days ago\n•\n30.5k\n•\n1.29k\ntencent/HY-MT1.5-1.8B\nUpdated\n1 day ago\n•\n847\n•\n365\nQwen/Qwen-Image-Edit-2511\nUpdated\n8 days ago\n•\n32.1k\n•\n583\nLiquidAI/LFM2-2.6B-Exp\nUpdated\n5 days ago\n•\n4.91k\n•\n272\nBrowse 2M+ models\nSpaces\nRunning\nFeatured\n3.29k\nWan2.2 Animate\n👁\n3.29k\nWan2.2

In [22]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [23]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a leading AI and machine learning (ML) community and collaboration platform dedicated to building the future of artificial intelligence. It serves as a central hub where ML engineers, scientists, researchers, and enthusiasts come together to share, explore, and experiment with open-source models, datasets, and AI applications.

At the heart of the AI revolution, Hugging Face empowers the next generation of AI professionals to create, collaborate, and innovate ethically and openly.

---

## What We Offer

- **Models**: Access and contribute to over **2 million open-source ML models** ranging across modalities — text, image, video, audio, and 3D. Stay on the cutting edge by using or improving models like state-of-the-art language transformers and image generators.

- **Datasets**: Discover and share from **over 500,000 public datasets** curated by the community, covering a diverse range of ML tasks and domains.

- **Spaces**: Host and deploy ML applications interactively in the cloud with **over 1 million applications** running, enabling rapid prototyping and sharing with others.

- **Community & Collaboration**: Join a fast-growing, vibrant community to learn, discuss, and advance AI technology together. Build your own ML portfolio and showcase your projects.

- **Docs & Resources**: Comprehensive documentation and tutorials help users, from beginners to experts, onboard and excel at ML development.

- **Enterprise Solutions**: Tailored plans providing advanced AI platforms for organizations with enterprise-grade security, access controls, dedicated support, and scalable compute options.

---

## For Enterprise and Teams

Hugging Face's **Team & Enterprise Hub** is designed to support organizations scaling AI projects securely and efficiently.

- **Starts at $20/user/month** for team subscriptions.
- Features include:
  - Single Sign-On (SSO) integration
  - Granular access management with Resource Groups
  - Detailed audit logs and analytics dashboard
  - Advanced compute options including ZeroGPU quotas
  - Private datasets viewers and additional private storage
  - Centralized token and billing management for inference providers
  - Enterprise-grade security compliance

This enables organizations to manage AI repositories, collaborate across teams, and accelerate AI innovation while maintaining full control and compliance.

---

## Company Culture

- **Open & Ethical AI**: Hugging Face champions transparency and openness in AI development to cultivate ethical AI innovations.
- **Community Driven**: The platform thrives on contributions from a passionate and diverse global ML community.
- **Innovation at the Edge**: The in-house science team pushes boundaries in AI and ML research.
- **Collaborative Environment**: Collaboration and sharing are core values, fostering growth and learning within the AI ecosystem.
- **Diverse Modalities**: Supporting a broad spectrum of ML work including language, vision, audio, and 3D modeling.

---

## Careers @ Hugging Face

Looking to join a leading AI company? Hugging Face offers opportunities for:

- Machine Learning Engineers
- Research Scientists
- Software Developers
- Data Scientists
- Community Managers
- Enterprise Solutions Specialists

Join a talented and passionate team working remotely and in offices, pushing the frontiers of AI technology. Be part of a mission to build an open and ethical AI future.

Explore current openings and apply via the Hugging Face Careers page.

---

## Join the AI Future with Hugging Face

- Collaborate on the Hugging Face Hub — the premier platform to share and discover machine learning models and datasets.
- Build your AI projects faster with an extensive ecosystem and open-source libraries.
- Leverage enterprise tools to ensure your organization’s AI efforts are secure, scalable, and effective.
- Become part of a vibrant, global AI community driving the future of technology.

**Get Started Today!**

Visit: [huggingface.co](https://huggingface.co)  
Sign up for free or explore enterprise options to accelerate your AI journey.

---

### Brand Identity

- Signature colors: Yellow (#FFD21E), Orange (#FF9D00), Gray (#6B7280)
- Recognized for fostering an inclusive, open, and innovative AI community

---

### Connect with Us

- GitHub | Twitter | LinkedIn | Discord  
- Browse models, datasets, and Spaces on the Hugging Face Hub  
- Join discussions and contribute via the community forums  

---

*Hugging Face – The AI community building the future.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [24]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [25]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face: The AI Community Building the Future

---

## About Us

Hugging Face is a vibrant and fast-growing community platform dedicated to fostering collaboration in the machine learning (ML) and artificial intelligence (AI) space. We provide a central hub where ML engineers, scientists, and AI enthusiasts can create, share, discover, and experiment with open-source models, datasets, and applications.

Our mission is to empower the next generation of AI practitioners to build an open, ethical, and accessible AI future by providing the tools and infrastructure needed to collaborate and innovate at speed.

---

## What We Offer

### Hugging Face Hub
- **2 Million+ Machine Learning Models**: A vast repository of state-of-the-art models across various AI modalities including text, image, video, audio, and even 3D.
- **500,000+ Datasets**: Access and contribute to richly curated datasets to fuel your ML projects.
- **1 Million+ Applications (Spaces)**: Interactive AI apps and collaborative projects built by the community, offering everything from image generation and editing to 3D modeling.

### Collaboration Platform
- Host and collaborate on unlimited public models, datasets, and applications.
- Share your work and build a recognized ML profile.
- Leverage the open-source HF stack for faster innovation.

### Enterprise Solutions
- Paid Compute and Enterprise-grade services designed for teams and organizations.
- Features include advanced platform capabilities, enterprise-grade security, access controls, and dedicated support.
- Pricing plans start at $20/month.

---

## Our Culture

- **Community-First**: We believe in building AI together through openness and inclusivity.
- **Ethical AI**: We are committed to nurturing an ethical future for AI technologies.
- **Collaboration & Sharing**: Encouraging transparent sharing of ideas and projects to rapidly advance the field.
- **Innovation Driven**: Empowering users to explore cutting-edge research and applications seamlessly.

---

## Our Customers & Users

- Individual AI researchers and machine learning engineers building and sharing models.
- Academic institutions seeking open and collaborative ML tools.
- Enterprises requiring secure and scalable AI development platforms.
- Developers creating innovative AI-powered applications.

---

## Careers at Hugging Face

Join a passionate team dedicated to advancing AI through community and open-source collaboration. 

#### Open Roles Include
- Machine Learning Engineers
- Software Developers
- Community Managers
- Enterprise Sales and Support

At Hugging Face, you will work alongside leading ML experts in an inclusive, creative, and supportive environment that values impact and innovation.

---

## Get Involved

- **Explore AI Applications:** Dive into thousands of community-built AI apps and models.
- **Create and Share:** Build your portfolio, host projects, and contribute to datasets or models.
- **Join the Community:** Collaborate, learn, and grow with professionals and enthusiasts worldwide.
- **Accelerate Your ML Projects:** Utilize our enterprise offerings for scalable compute and security.

**Sign up today and be part of the future of AI!**

[Visit Hugging Face](https://huggingface.co)

---

## Brand Highlights

Our vibrant identity is reflected in our signature colors:
- Bright Yellow: #FFD21E
- Orange Accent: #FF9D00
- Modern Gray: #6B7280

Our logos and assets are openly available to support community and partner use.

---

Hugging Face - Where the Machine Learning Community Builds the Future.

In [26]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the vibrant AI community building the future of machine learning. As a leading collaboration platform, Hugging Face empowers researchers, engineers, data scientists, and AI enthusiasts worldwide to create, share, and discover machine learning models, datasets, and applications.

The Hugging Face Hub serves as a central open-source space where millions of users explore over 2 million models and hundreds of thousands of datasets, spanning modalities such as text, image, audio, video, and even 3D.

---

## Platform Highlights

- **Models**: Access and contribute to 2M+ pre-trained machine learning models—enabling rapid experimentation and deployment.
- **Datasets**: Browse from 500,000+ datasets curated for diverse AI tasks.
- **Spaces**: Host and run interactive AI apps and demos in the cloud.
- **Community Driven**: Foster collaboration and knowledge exchange within one of the largest ML communities.
- **Open Source Stack**: Build faster using Hugging Face’s comprehensive open source tools.
- **Multi-Modality**: Support for text, vision, audio, video, and 3D models encourages innovation across AI disciplines.

---

## Customers & Use Cases

Hugging Face serves a diverse user base including:

- Independent Machine Learning Engineers and Researchers building portfolios and sharing projects.
- AI Teams in industry leveraging the platform’s collaboration tools and enterprise-grade security.
- Organizations requiring scalable Compute and Enterprise solutions for AI development and deployment.
- Open source contributors maintaining some of the most widely used ML libraries.

The platform accelerates research and production workflows across natural language processing, computer vision, audio processing, and creative AI applications such as image editing, 3D generation, and more.

---

## Company Culture

Hugging Face thrives on transparency, openness, and community. They nurture:

- **Ethical AI**: Commitment to building an open and ethical AI future powered by collaboration.
- **Community Empowerment**: Enabling the next generation of AI practitioners to learn, share and innovate.
- **Innovation & Accessibility**: Democratizing AI by providing free access and paid options tailored for teams and enterprises.
- **Diversity & Inclusion**: Welcoming contributors from all backgrounds to grow the AI ecosystem together.

---

## Careers at Hugging Face

Join a fast-growing company at the forefront of machine learning technology. Hugging Face offers exciting opportunities to work on:

- Cutting edge ML infrastructure and open source tools.
- Building scalable cloud platforms for AI collaboration.
- Research and development on state-of-the-art models across modalities.
- Supporting a global and active AI community.
  
With a culture that encourages creativity, community impact, and continuous learning, Hugging Face is ideal for engineers, researchers, product managers, and AI enthusiasts passionate about shaping the future of AI.

---

## Get Started

- Explore AI apps, models, and datasets on the Hugging Face platform.
- Build your own portfolio by sharing your ML projects.
- Accelerate your AI development with paid Compute and Enterprise plans starting at $20/month.
- Join the community and contribute to the world's fastest growing ML hub.

Visit [huggingface.co](https://huggingface.co) to sign up and start collaborating today!

---

## Brand & Visual Identity

- Recognizable by their friendly "Hugging Face" logo and bright color palette (#FFD21E, #FF9D00, #6B7280).
- Commitment to open collaboration is reflected in their accessible resources and community-driven ethos.

---

### Hugging Face: The AI community building the future.  
Empowering open, ethical, and collaborative machine learning for everyone.

In [ ]:
import gradio as gr
import requests

class BrochureGenerator:
    """Unified brochure generator supporting multiple LLM providers."""
    
    def __init__(self):
        """Initialize all model clients."""
        # OpenAI client
        self.openai_client = OpenAI()
        
        # Ollama client (check if available)
        self.ollama_available = False
        try:
            requests.get("http://localhost:11434/", timeout=2)
            self.ollama_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
            self.ollama_available = True
        except:
            self.ollama_client = None
        
        # Model registry: maps display name to (client, model_name) tuple
        self.models = {
            "GPT": (self.openai_client, "gpt-4.1-mini"),
        }
        
        if self.ollama_available:
            self.models["Ollama"] = (self.ollama_client, "llama3.2")
    
    def stream_brochure(self, company_name, url, model_name):
        """
        Generate brochure with streaming support.
        
        Args:
            company_name: Name of the company
            url: Company website URL
            model_name: Display name of model (e.g., "GPT", "Ollama")
        
        Yields:
            str: Incremental response chunks
        """
        if model_name not in self.models:
            yield f"Error: Model '{model_name}' not available."
            return
        
        client, model = self.models[model_name]
        
        # Build messages using existing functions
        messages = [
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ]
        
        # Stream response
        stream = client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True
        )
        
        response = ""
        for chunk in stream:
            response += chunk.choices[0].delta.content or ''
            yield response
    
    def get_available_models(self):
        """Return list of available model names."""
        return list(self.models.keys())


# Initialize generator
generator = BrochureGenerator()

# Gradio interface
company_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(
    generator.get_available_models(), 
    label="Select model", 
    value="GPT"
)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=generator.stream_brochure,
    title="Company Brochure Generator", 
    inputs=[company_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
        ["EdDonner", "https://edwarddonner.com", "GPT"],
        ["HuggingFace", "https://huggingface.co", "GPT"]
    ], 
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>